In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from core.nn.timesfm import NewsTimesFM_2p5_Model
from transformers import AutoTokenizer
from safetensors.torch import load_file
import torch
import polars as pl
from core.training.data.dataset import TimesFMDataset, collate_fn
from clearml import InputModel
from core.nn.text_encoder.tokenizer import NewsTokenizerWrapper
from torch.utils.data import DataLoader
from core.training.data import TimesFMDataModule
from core.training.configs import DataConfig
from functools import partial
import plotly.express as px
import random
from clearml import Task, OutputModel
from transformers.models.modernbert.modeling_modernbert import ModernBertModel
import orjson
from pathlib import Path
from core.nn.timesfm.configs import ModernBertConfig, TimesFM_2p5_200M_Config

In [3]:
data_cfg = DataConfig(
    dataset_id="a9dc8daf2d89450b8812ae4cf78514a1",
    num_workers=8,
    batch_size=3,
    output_patch_len=20,
    input_patch_len=32,
    context_len=256,
)

In [4]:
datamodule = TimesFMDataModule(
    data_cfg=data_cfg,
    tokenizer_path=InputModel(
        model_id="c84c36cfe62b41e7b4c3c9217beff551"
    ).get_local_copy(),
)

In [5]:
datamodule.prepare_data()
datamodule.setup(stage="fit")

In [6]:
(data['inputs_ts'][..., -1].log().unsqueeze(-1) + data['targets_log_returns'].cumsum(-1)).exp()

NameError: name 'data' is not defined

In [6]:
train_loader = datamodule.train_dataloader()

In [7]:
dl = iter(train_loader)

In [9]:
batch = next(dl)
batch.keys()


dict_keys(['inputs_ts', 'inputs_log_returns', 'mask_ts', 'targets_log_returns', 'targets', 'inputs_text', 'mask_text'])

Collate batch size: 3
Collation time: 0.00038886070251464844


In [11]:
(batch["inputs_ts"][..., -1].log().unsqueeze(-1) + batch["targets_log_returns"].cumsum(-1)).exp()

tensor([[[334.0000, 333.6999, 334.2000, 335.0499, 335.7499, 335.4499, 335.9999,
          338.7499, 338.4501, 339.2500, 338.9000, 340.5999, 340.9000, 343.0000,
          342.4999, 343.1999, 342.3999, 344.4499, 344.6999, 344.9499],
         [349.3000, 349.5500, 324.3500, 318.2000, 319.3500, 324.0000, 323.7000,
          323.7000, 322.2001, 320.5500, 317.1000, 316.0500, 317.1499, 316.4500,
          316.2000, 319.3500, 316.8500, 318.1499, 317.9999, 316.1000],
         [323.0000, 323.3000, 324.2000, 327.4501, 329.8501, 330.4000, 331.1501,
          329.3500, 329.1001, 330.1500, 330.4001, 328.1001, 329.1001, 328.0500,
          328.6000, 329.2500, 328.8500, 330.3500, 333.5501, 332.6500],
         [328.8500, 327.3500, 327.7000, 327.5499, 328.4999, 328.8999, 319.0000,
          318.0500, 317.1499, 319.5499, 318.7999, 319.4000, 319.0999, 319.2498,
          316.8499, 316.2499, 316.2499, 313.9500, 312.8999, 312.1999],
         [312.5000, 312.8000, 310.1001, 305.2001, 307.4502, 305.5001, 306.05

In [ ]:
news_timesfm = NewsTimesFM_2p5_Model.from_pretrained(
    "/home/pomelk1n/Workspace/finam-forecast/models/new-timesfm-2p5-small-bert",
    dtype=torch.bfloat16,
)

Missing keys : ['text_token_weight_proj.weight']
Unexpected keys : ['stacked_xf.0.attn.rotary_position_embedding.max_seq_len', 'stacked_xf.0.cross_attn.text_rotary_position_embedding.max_seq_len', 'stacked_xf.0.cross_attn.ts_rotary_position_embedding.max_seq_len', 'stacked_xf.1.attn.rotary_position_embedding.max_seq_len', 'stacked_xf.1.cross_attn.text_rotary_position_embedding.max_seq_len', 'stacked_xf.1.cross_attn.ts_rotary_position_embedding.max_seq_len', 'stacked_xf.2.attn.rotary_position_embedding.max_seq_len', 'stacked_xf.2.cross_attn.text_rotary_position_embedding.max_seq_len', 'stacked_xf.2.cross_attn.ts_rotary_position_embedding.max_seq_len', 'stacked_xf.3.attn.rotary_position_embedding.max_seq_len', 'stacked_xf.3.cross_attn.text_rotary_position_embedding.max_seq_len', 'stacked_xf.3.cross_attn.ts_rotary_position_embedding.max_seq_len', 'stacked_xf.4.attn.rotary_position_embedding.max_seq_len', 'stacked_xf.4.cross_attn.text_rotary_position_embedding.max_seq_len', 'stacked_xf.4.c

In [ ]:
inputs_ts = batch["inputs_log_returns"].to(torch.bfloat16)
mask_ts = batch["mask_ts"].to(torch.bfloat16)
targets = batch["targets_log_returns"].to(torch.bfloat16)

inputs_text = batch["inputs_text"]
mask_text = batch["mask_text"]

output = news_timesfm.forecast(
    inputs_ts=inputs_ts,
    mask_ts=mask_ts,
    inputs_text=inputs_text,
    mask_text=mask_text,
    targets=targets,
)
output

{'normalized_output': tensor([[[ 0.8672,  1.1641,  0.6523,  ...,  2.4375,  1.7109,  1.2266],
          [-1.3438, -0.7031, -1.0859,  ...,  1.5859,  0.4531,  0.8555],
          [ 0.6797,  0.9688,  0.4688,  ...,  1.8672,  0.9219,  0.7383],
          ...,
          [ 1.2969,  1.5938,  1.0234,  ...,  1.7109,  1.4688,  1.1406],
          [ 0.8984,  1.3984,  0.5625,  ...,  1.8984,  1.6406,  1.8750],
          [ 1.5859,  1.9062,  1.2188,  ...,  2.0000,  1.5781,  1.6094]]],
        dtype=torch.bfloat16, grad_fn=<SelectBackward0>),
 'output': tensor([[[152., 153., 151.,  ..., 156., 154., 153.],
          [146., 148., 147.,  ..., 155., 151., 153.],
          [151., 152., 151.,  ..., 155., 152., 151.],
          ...,
          [160., 161., 158.,  ..., 162., 161., 159.],
          [159., 163., 157.,  ..., 166., 165., 166.],
          [171., 174., 167.,  ..., 175., 171., 171.]]], dtype=torch.bfloat16,
        grad_fn=<AddBackward0>),
 'normalized_target': tensor([[[ 1.6250,  1.6250,  1.2969,  ...,  

In [ ]:
output["normalized_output"].mean(-1), output["normalized_target"].mean(-1)

(tensor([[ 0.5430, -0.1191,  0.3164,  0.0055,  0.0623,  0.9258,  1.0625,  1.3594]],
        dtype=torch.bfloat16, grad_fn=<MeanBackward1>),
 tensor([[0.1934, 0.7930, 2.2656, 2.9062, 4.2188, 2.2188, 1.2891, 0.1699]],
        dtype=torch.bfloat16))

In [ ]:
output["normalized_output"].mean(-1), output["normalized_target"].mean(-1)

(tensor([[-0.1641,  0.6250,  0.2773,  0.2373, -0.1089,  0.0503,  0.2246,  0.6836]],
        dtype=torch.bfloat16, grad_fn=<MeanBackward1>),
 tensor([[ 0.2119,  0.1494, -0.3457, -0.2852,  0.3398,  1.1250,  2.0000,  2.3125]],
        dtype=torch.bfloat16))

In [ ]:
torch.nn.functional.huber_loss(
    output["normalized_output"], output["normalized_target"], delta=2.0
)

tensor(2.5938, dtype=torch.bfloat16, grad_fn=<HuberLossBackward0>)

In [39]:
376 * 376

141376

In [ ]:
torch.nn.functional.mse_loss(output["normalized_output"], output["nb ormalized_target"])

tensor(2.1562, dtype=torch.bfloat16, grad_fn=<MseLossBackward0>)

In [ ]:
t = batch["mask_text"].size(-1)
seq_lens = (batch["mask_text"].view(-1, t).bool()).sum(-1)
unpadded_seqs: list[torch.Tensor] = []
for seq, seq_len in zip(batch["inputs_text"].view(-1, t), seq_lens):
    unpadded_seqs.append(seq[:seq_len].unsqueeze(0))

In [24]:
unpadded_seqs

[tensor([[50281, 50368, 50282, 50368, 50282, 50368, 50282, 50368, 50282, 50368,
          50282, 50368, 50282, 50368, 50282, 50368, 50282, 50368, 50282, 50368,
          50282, 50368, 50282, 50368, 50282, 50368, 50282, 50368, 50282, 50368,
          50282, 50368, 50282, 50368, 50282, 50368, 50282, 50368, 50282, 50368,
          50282, 50368, 50282, 50368, 50282, 50368, 50282, 50368, 50282, 50368,
          50282, 50368, 50282, 50368, 50282, 50368, 50282, 50368, 50282, 50368,
          50282, 50368, 50282, 50368, 50282]]),
 tensor([[50281, 50368, 50282, 50368, 50282, 50368, 50282, 50368, 50282, 50368,
          50282, 50368, 50282, 50368, 50282, 50368, 50282, 50368, 50282, 50368,
          50282, 50368, 50282, 50368, 50282, 50368, 50282, 50368, 50282, 50368,
          50282, 50368, 50282, 50368, 50282, 50368, 50282, 50368, 50282, 50368,
          50282, 50368, 50282, 50368, 50282, 50368, 50282, 50368, 50282, 50368,
          50282, 50368, 50282, 50368, 50282, 50368, 50282, 50368, 50282,

In [ ]:
inputs_ts = batch["inputs_ts"].to(torch.bfloat16)
mask_ts = batch["mask_ts"].to(torch.bfloat16)

inputs_text = batch["inputs_text"]
mask_text = batch["mask_text"]

output = news_timesfm.forecast(
    inputs_ts=inputs_ts,
    mask_ts=mask_ts,
    inputs_text=inputs_text,
    mask_text=mask_text,
)
output

tensor([[[67.5000, 67.5000, 67.5000,  ..., 68.5000, 68.0000, 68.0000],
         [69.5000, 70.0000, 69.0000,  ..., 71.0000, 70.0000, 69.5000],
         [67.5000, 68.0000, 67.5000,  ..., 70.0000, 68.5000, 69.0000],
         ...,
         [23.3750, 33.0000, 18.7500,  ..., 51.2500, 42.0000, 41.7500],
         [23.7500, 31.3750, 17.0000,  ..., 44.2500, 37.2500, 36.5000],
         [27.2500, 34.5000, 22.6250,  ..., 48.0000, 42.0000, 43.2500]]],
       dtype=torch.bfloat16, grad_fn=<AddBackward0>)

In [10]:
batch["targets"]

tensor([[[23.3600, 22.4400, 23.2000,  ..., 32.2700, 32.3900, 32.7900],
         [25.0600, 25.2400, 25.1800,  ..., 39.0900, 39.5100, 39.9800],
         [25.4600, 26.0800, 26.7200,  ..., 41.4300, 42.0900, 41.8500],
         ...,
         [38.6700, 38.5100, 37.8200,  ..., 57.7100, 58.1300, 55.0800],
         [37.4500, 37.8000, 38.0100,  ..., 53.7800, 52.3300, 56.7300],
         [38.0700, 37.7200, 37.6100,  ..., 46.6500, 46.4500, 49.5300]]])

In [15]:
loss_fn = torch.nn.MSELoss()

outpatch_len = output.shape[-1]
total_loss = loss_fn(output, batch["targets"].to(torch.bfloat16))
loss_temp = 0.0
for i in range(outpatch_len):
    loss = loss_fn(output[..., i], batch["targets"][..., i].to(torch.bfloat16))
    loss_temp += loss
    print(f"Step {i}, Loss: {loss.item()}")
total_loss, loss_temp / outpatch_len

Step 0, Loss: 3.5625
Step 1, Loss: 53.75
Step 2, Loss: 27.875
Step 3, Loss: 79.5
Step 4, Loss: 136.0
Step 5, Loss: 146.0
Step 6, Loss: 137.0
Step 7, Loss: 142.0
Step 8, Loss: 128.0
Step 9, Loss: 81.5
Step 10, Loss: 123.0
Step 11, Loss: 107.0
Step 12, Loss: 70.0
Step 13, Loss: 119.5
Step 14, Loss: 129.0
Step 15, Loss: 81.0
Step 16, Loss: 79.5
Step 17, Loss: 109.0
Step 18, Loss: 235.0
Step 19, Loss: 106.0
Step 20, Loss: 111.0
Step 21, Loss: 376.0
Step 22, Loss: 226.0
Step 23, Loss: 290.0
Step 24, Loss: 119.0
Step 25, Loss: 490.0
Step 26, Loss: 272.0
Step 27, Loss: 225.0
Step 28, Loss: 148.0
Step 29, Loss: 472.0
Step 30, Loss: 107.5
Step 31, Loss: 117.5
Step 32, Loss: 91.0
Step 33, Loss: 255.0
Step 34, Loss: 148.0
Step 35, Loss: 288.0
Step 36, Loss: 206.0
Step 37, Loss: 416.0
Step 38, Loss: 446.0
Step 39, Loss: 320.0
Step 40, Loss: 326.0
Step 41, Loss: 692.0
Step 42, Loss: 520.0
Step 43, Loss: 836.0
Step 44, Loss: 492.0
Step 45, Loss: 338.0
Step 46, Loss: 344.0
Step 47, Loss: 576.0
Step 4

(tensor(402., dtype=torch.bfloat16, grad_fn=<MseLossBackward0>),
 tensor(400., dtype=torch.bfloat16, grad_fn=<DivBackward0>))

In [ ]:
batch_size = output.size(0)
batch_idx = random.randint(0, batch_size - 1)
last_pred = output[batch_idx, -2:, :].view(-1).detach().cpu().float().numpy()
last_target = (
    batch["targets"][batch_idx, -2:, :].view(-1).detach().cpu().float().numpy()
)

fig = px.line()
fig.add_scatter(y=last_pred, mode="lines+markers", name="Prediction")
fig.add_scatter(y=last_target, mode="lines+markers", name="Target")
fig.show()

In [32]:
output[batch_idx, -2:].shape

torch.Size([2, 128])